# Linear Regression với Lag Features
**Thành viên 1 — Model tự chọn**

## Mô tả model
Linear Regression sử dụng **lag features** (giá đóng cửa của N ngày trước) kết hợp với **technical indicators** đã được tính sẵn (MA, RSI, MACD, Bollinger Bands) để dự báo giá đóng cửa ngày tiếp theo.

### Chiến lược tránh data leakage
- **Target:** `close[t]` — giá đóng cửa ngày t
- **Features:** dữ liệu từ ngày `t-1` trở về trước  
  - `close_lag_1..5`: giá đóng cửa 1–5 ngày trước  
  - `*_prev`: tất cả indicators tính đến ngày t-1 (shift=1)  
  - `rolling_std_5/20`: biến động (volatility) lịch sử

### Pipeline
```
Load CSV splits → Tạo lag features → StandardScaler (fit on train only)
  → LinearRegression.fit(X_train, y_train)
  → predict(X_test) → RMSE / MAE / MAPE / R²
```

**Kết quả:** 10 biểu đồ (5 mã × 2 splits) + 1 CSV tổng hợp metrics

---
## 0 — Imports & Cấu hình

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
ROOT         = Path('..').resolve()
SPLITS_DIR   = ROOT / 'data' / 'processed' / 'splits'
RESULTS_DIR  = ROOT / 'results' / 'linear_regression'
PLOTS_DIR    = RESULTS_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'Linear Regression'
TICKERS    = ['VCB', 'FPT', 'HPG', 'VIC', 'VNM']
SPLITS     = ['70_30', '80_20']

# Lag parameters
LAG_DAYS       = [1, 2, 3, 5, 10]   # lag của close price
ROLLING_WINS   = [5, 20]             # rolling std windows

# Technical indicator columns (đã có trong featured data)
TECH_COLS = [
    'open', 'high', 'low', 'volume',
    'ma_5', 'ma_20', 'ma_50',
    'rsi_14',
    'macd', 'macd_signal', 'macd_hist',
    'bb_upper', 'bb_middle', 'bb_lower',
]

# Đọc split info
split_info = json.loads((SPLITS_DIR / 'split_info.json').read_text())
print(f'Results dir: {RESULTS_DIR}')
print(f'Tickers: {TICKERS}')
print(f'Splits: {SPLITS}')

---
## 1 — Hàm tiện ích

In [ ]:
def load_split(ticker: str, split: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load train/test CSV cho một mã và tỷ lệ split."""
    base  = SPLITS_DIR / split
    train = pd.read_csv(base / f'{ticker}_train.csv', parse_dates=['date'])
    test  = pd.read_csv(base / f'{ticker}_test.csv',  parse_dates=['date'])
    return (train.sort_values('date').reset_index(drop=True),
            test.sort_values('date').reset_index(drop=True))


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """RMSE, MAE, MAPE, R² — chuẩn báo cáo nhóm."""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    mask = y_true != 0
    return {
        'RMSE':     float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE':      float(mean_absolute_error(y_true, y_pred)),
        'MAPE (%)': float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100),
        'R²':       float(r2_score(y_true, y_pred)),
    }


def plot_predictions(dates, y_true, y_pred, ticker, split, metrics, save=True):
    """Vẽ Actual vs Predicted, lưu PNG."""
    fig, ax = plt.subplots(figsize=(14, 4.5))
    ax.plot(dates, y_true, label='Actual',    color='#2563EB', lw=1.3, alpha=0.9)
    ax.plot(dates, y_pred, label='Predicted', color='#DC2626', lw=1.3, alpha=0.9, ls='--')

    subtitle = (f"RMSE={metrics['RMSE']:,.2f}  "
                f"MAE={metrics['MAE']:,.2f}  "
                f"MAPE={metrics['MAPE (%)']:.2f}%  "
                f"R²={metrics['R²']:.4f}")
    ax.set_title(f"{MODEL_NAME} — {ticker}  (Split {split.replace('_', '/')})\n{subtitle}", pad=10)
    ax.set_xlabel('Ngày')
    ax.set_ylabel('Giá đóng cửa (VND × nghìn)')
    ax.legend(loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=35, ha='right')
    ax.grid(axis='y', ls=':', alpha=0.5)
    plt.tight_layout()

    if save:
        fname = PLOTS_DIR / f'{ticker}_{split}_linear_regression.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'    Saved → {fname.relative_to(ROOT)}')
    plt.show()

---
## 2 — Feature Engineering

Hàm `create_lag_features` kết hợp train+test trước khi tính lag  
để hàng đầu tiên của test có đủ dữ liệu từ cuối train.

In [ ]:
def create_lag_features(combined: pd.DataFrame) -> pd.DataFrame:
    """
    Thêm lag features vào DataFrame đã ghép (train + test).

    Features được tạo (tất cả shift=1 để không dùng giá trị ngày hiện tại):
      - close_lag_1 … close_lag_10  : giá đóng cửa N ngày trước
      - rolling_std_5, rolling_std_20: độ lệch chuẩn rolling của close
      - <col>_prev                  : technical indicators ngày trước

    Target: close[t]  (không shift)
    """
    df = combined.copy().sort_values('date').reset_index(drop=True)

    # Lag của close price
    for lag in LAG_DAYS:
        df[f'close_lag_{lag}'] = df['close'].shift(lag)

    # Rolling volatility (tính từ close đã shift 1 để tránh leakage)
    shifted_close = df['close'].shift(1)
    for w in ROLLING_WINS:
        df[f'rolling_std_{w}'] = shifted_close.rolling(w).std()

    # Technical indicators ngày t-1
    for col in TECH_COLS:
        if col in df.columns:
            df[f'{col}_prev'] = df[col].shift(1)

    return df


def get_feature_cols(df: pd.DataFrame) -> list[str]:
    """Trả về danh sách cột dùng làm features (loại trừ target và raw OHLCV)."""
    skip = {'date', 'open', 'high', 'low', 'close', 'volume',
             'ma_5', 'ma_20', 'ma_50',
             'rsi_14', 'macd', 'macd_signal', 'macd_hist',
             'bb_upper', 'bb_middle', 'bb_lower'}
    return [c for c in df.columns if c not in skip]

### Kiểm tra feature list với một mã mẫu

In [ ]:
# Xem thử features được tạo ra
_train, _test = load_split('VCB', '70_30')
_combined     = create_lag_features(pd.concat([_train, _test], ignore_index=True))
_feats        = get_feature_cols(_combined)

print(f'Số features: {len(_feats)}')
for f in _feats:
    print(f'  {f}')

---
## 3 — Train & Predict

In [ ]:
def train_and_predict_lr(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> dict:
    """
    Train Linear Regression trên train_df, trả về predictions cho test_df.

    Quy trình:
      1. Ghép train+test → tính lag features
      2. Tách lại train/test theo ngày
      3. Loại bỏ hàng NaN (warm-up lag)
      4. StandardScaler fit on train only
      5. LinearRegression.fit → predict

    Returns dict với keys: y_pred, y_true, dates, model, scaler, feature_cols
    """
    cut_date = train_df['date'].iloc[-1]

    # Tạo lag features trên toàn bộ chuỗi (tránh cold-start cho test)
    combined     = create_lag_features(pd.concat([train_df, test_df], ignore_index=True))
    feature_cols = get_feature_cols(combined)

    # Tách train / test theo ngày gốc
    train_feat = combined[combined['date'] <= cut_date].dropna(subset=feature_cols)
    test_feat  = combined[combined['date'] >  cut_date].dropna(subset=feature_cols)

    X_train = train_feat[feature_cols].values
    y_train = train_feat['close'].values
    X_test  = test_feat[feature_cols].values

    # Chuẩn hoá features (fit chỉ trên train)
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    # Huấn luyện
    model = LinearRegression()
    model.fit(X_train, y_train)

    return {
        'y_pred':       model.predict(X_test),
        'y_true':       test_feat['close'].values,
        'dates':        test_feat['date'].values,
        'model':        model,
        'scaler':       scaler,
        'feature_cols': feature_cols,
    }

---
## 4 — Chạy toàn bộ: 5 mã × 2 splits = 10 biểu đồ

In [ ]:
all_results = []

for split in SPLITS:
    print(f"\n{'═'*60}")
    print(f"  SPLIT  {split.replace('_', '/')}")
    print(f"{'═'*60}")

    for ticker in TICKERS:
        print(f"\n  ▶  {ticker}")

        train_df, test_df = load_split(ticker, split)
        print(f"    Train: {len(train_df)} rows  |  Test: {len(test_df)} rows")

        res = train_and_predict_lr(train_df, test_df)

        metrics = compute_metrics(res['y_true'], res['y_pred'])
        print(f"    RMSE={metrics['RMSE']:>10,.2f}  "
              f"MAE={metrics['MAE']:>10,.2f}  "
              f"MAPE={metrics['MAPE (%)']:>6.2f}%  "
              f"R²={metrics['R²']:.4f}")

        plot_predictions(res['dates'], res['y_true'], res['y_pred'], ticker, split, metrics)

        all_results.append({
            'Ticker': ticker,
            'Split': split,
            'Model': MODEL_NAME,
            **metrics,
        })

---
## 5 — Bảng tổng hợp kết quả

In [ ]:
results_df = (
    pd.DataFrame(all_results)
    [['Ticker', 'Split', 'Model', 'RMSE', 'MAE', 'MAPE (%)', 'R²']]
    .sort_values(['Split', 'Ticker'])
    .reset_index(drop=True)
)
results_df[['RMSE', 'MAE']] = results_df[['RMSE', 'MAE']].round(2)
results_df['MAPE (%)'] = results_df['MAPE (%)'].round(3)
results_df['R²']       = results_df['R²'].round(4)

print(results_df.to_string(index=False))

out_csv = RESULTS_DIR / 'linear_regression_results.csv'
results_df.to_csv(out_csv, index=False)
print(f'\nĐã lưu → {out_csv.relative_to(ROOT)}')

### Pivot theo RMSE — so sánh nhanh giữa các mã

In [ ]:
pivot_rmse = results_df.pivot_table(index='Ticker', columns='Split', values='RMSE')
pivot_mae  = results_df.pivot_table(index='Ticker', columns='Split', values='MAE')

print('RMSE (VND × nghìn):')
print(pivot_rmse.round(2).to_string())
print('\nMAE (VND × nghìn):')
print(pivot_mae.round(2).to_string())

---
## 6 — Phân tích hệ số hồi quy

Xem đặc trưng nào quan trọng nhất với mã VCB (70/30 split).

In [ ]:
# Huấn luyện lại VCB 70_30 để lấy model object
_train, _test = load_split('VCB', '70_30')
_res = train_and_predict_lr(_train, _test)

coef_df = (
    pd.DataFrame({'Feature': _res['feature_cols'], 'Coefficient': _res['model'].coef_})
    .assign(AbsCoef=lambda x: x['Coefficient'].abs())
    .sort_values('AbsCoef', ascending=False)
    .head(15)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2563EB' if c >= 0 else '#DC2626' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'][::-1], coef_df['Coefficient'][::-1], color=colors[::-1])
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Top 15 Feature Coefficients — VCB (70/30)\n(Xanh = dương, Đỏ = âm)')
ax.set_xlabel('Hệ số hồi quy (đã chuẩn hoá)')
plt.tight_layout()
coef_path = PLOTS_DIR / 'VCB_70_30_feature_importance.png'
fig.savefig(coef_path, dpi=150, bbox_inches='tight')
plt.show()
print(coef_df[['Feature', 'Coefficient']].to_string(index=False))

---
## 7 — Nhận xét & Phân tích

### Ưu điểm của Linear Regression + Lag Features
- **Tốc độ nhanh**: huấn luyện trong vài mili-giây
- **Diễn giải được**: hệ số hồi quy cho thấy tầm quan trọng của từng feature
- **Baseline mạnh**: với time series dạng random walk, lag 1 thường chiếm ưu thế

### Hạn chế
- Giả định quan hệ **tuyến tính** giữa features và target — không nắm bắt được phi tuyến
- Không mô hình hoá được phụ thuộc **dài hạn** (LSTM/GRU làm tốt hơn)
- Không có cơ chế xử lý **tính mùa vụ** (Prophet xử lý tốt hơn)

### Gợi ý khi viết báo cáo (Chương 4)
- Trình bày công thức: $\hat{y}_t = \beta_0 + \sum_{i=1}^{k} \beta_i x_{t-1,i}$
- Giải thích feature `close_lag_1` thường có hệ số lớn nhất (random walk property)
- So sánh với mô hình Naive baseline (predict = giá hôm qua) để thấy LR cải thiện bao nhiêu